# 06 Eustace_GSE65858_validation

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

In [ ]:
EUSTACE_GENES = [
    "ALDOA", "ANGPTL4", "ANLN", "BNC1", "MRGBP", "CA9", "CDKN3", "COL4A6",
    "DCBLD1", "ENO1", "FAM83B", "FOSL1", "GNAI1", "HILPDA", "KCTD11", "KRT17",
    "LDHA", "MRPS17", "P4HA1", "PGAM1", "PGK1", "SDC1", "SLC16A1", "SLC2A1",
    "TPI1", "VEGFA",
]
CONTROL_GENES = ["GNB2L1", "B2M", "RPL11"]
print("Eustace panel:", len(EUSTACE_GENES), "genes | Controls:", CONTROL_GENES)

In [ ]:
probe_to_symbol = {}
with open("GPL10558_HumanHT-12_V4_0_R2_15002873_B.txt", "r", encoding="latin-1") as f:
    for line in f:
        if line.startswith("Species\tSource\tSearch_Key"):
            header = line.strip().split("\t")
            break
    probe_idx = header.index("Probe_Id")
    symbol_idx = header.index("Symbol")
    for line in f:
        if line.startswith("[Controls]") or line.strip() == "":
            break
        fields = line.rstrip("\n").split("\t")
        if len(fields) <= max(probe_idx, symbol_idx):
            continue
        pid, sym = fields[probe_idx].strip(), fields[symbol_idx].strip()
        if pid and sym:
            probe_to_symbol[pid] = sym

needed_genes = EUSTACE_GENES + CONTROL_GENES
ALIAS_FIXES = {"MRGBP": "C20orf20"}

symbol_to_probe_ids = defaultdict(list)
for pid, sym in probe_to_symbol.items():
    if sym in needed_genes:
        symbol_to_probe_ids[sym].append(pid)
    for orig, alias in ALIAS_FIXES.items():
        if sym == alias and orig in needed_genes:
            symbol_to_probe_ids[orig].append(pid)

for g in needed_genes:
    if g not in symbol_to_probe_ids:
        print("Genuinely unresolved:", g)

In [ ]:
wb = openpyxl.load_workbook("GSE65858.xlsx", data_only=True, read_only=True)
ws = wb["Sheet1"]

header_row = list(ws.iter_rows(min_row=105, max_row=105, values_only=True))[0]
sample_ids = [str(v) for v in header_row[1:] if v]
needed_probe_ids = set(p for probes in symbol_to_probe_ids.values() for p in probes)

expression_rows = {}
for row in ws.iter_rows(min_row=106, max_row=ws.max_row, values_only=True):
    if row[0] in needed_probe_ids:
        expression_rows[row[0]] = row[1:1+len(sample_ids)]

gene_expr = {}
for gene, probes in symbol_to_probe_ids.items():
    vals = [expression_rows[p] for p in probes if p in expression_rows]
    if vals:
        gene_expr[gene] = np.array(vals, dtype=float).mean(axis=0)

expr_df = pd.DataFrame(gene_expr, index=sample_ids)
print("Gene-level expression matrix:", expr_df.shape)

In [ ]:
def get_row_values(row_idx, n_samples):
    row = list(ws.iter_rows(min_row=row_idx, max_row=row_idx, max_col=1+n_samples, values_only=True))[0]
    vals = []
    for v in row[1:]:
        if v is None:
            vals.append(None); continue
        v = str(v)
        vals.append(v.split(":", 1)[1].strip() if ":" in v else v)
    return vals

n = len(sample_ids)
os_time = get_row_values(51, n)
os_event = get_row_values(52, n)

clin = pd.DataFrame({"Sample": sample_ids, "OS_time_days": os_time, "OS_event_raw": os_event}).set_index("Sample")
clin["OS_time_days"] = pd.to_numeric(clin["OS_time_days"], errors="coerce")
clin["OS_months"] = clin["OS_time_days"] / 30.44
clin["Event"] = clin["OS_event_raw"].map({"TRUE": 1, "FALSE": 0})
clin = clin.dropna(subset=["OS_months", "Event"])
clin = clin[clin["OS_months"] > 0]

data = expr_df.join(clin[["OS_months", "Event"]], how="inner")
print("Patients with expression + survival:", data.shape[0], "| events:", int(data["Event"].sum()))

In [ ]:
EUSTACE_GENES_AVAILABLE = [g for g in EUSTACE_GENES if g in data.columns]
print("Panel genes actually used:", len(EUSTACE_GENES_AVAILABLE), "of", len(EUSTACE_GENES))

def zscore(s):
    return (s - s.mean()) / s.std()

panel_z = data[EUSTACE_GENES_AVAILABLE].apply(zscore)
control_z = data[CONTROL_GENES].apply(zscore)

data["EustaceScore_c"] = panel_z.median(axis=1) - control_z.mean(axis=1)
data["EustaceScore_c"] = data["EustaceScore_c"] - data["EustaceScore_c"].mean()

In [ ]:
d = data[["OS_months", "Event", "EustaceScore_c"]].dropna()
cph = CoxPHFitter()
cph.fit(d, duration_col="OS_months", event_col="Event")
cph.print_summary()

summ = cph.summary.loc["EustaceScore_c"]
hr, lo, hi, pval = np.exp(summ["coef"]), np.exp(summ["coef lower 95%"]), np.exp(summ["coef upper 95%"]), summ["p"]
cidx = cph.concordance_index_
print(f"\nHR={hr:.3f} (95% CI {lo:.3f}-{hi:.3f}), p={pval:.4g}, C-index={cidx:.3f}")

In [ ]:
med = d["EustaceScore_c"].median()
high, low = d[d["EustaceScore_c"] >= med], d[d["EustaceScore_c"] < med]
lr_full = logrank_test(high["OS_months"], low["OS_months"], high["Event"], low["Event"])
print("Log-rank p (full follow-up):", lr_full.p_value)

plt.figure(figsize=(7, 5.5))
KaplanMeierFitter().fit(high["OS_months"], high["Event"], label=f"High (n={len(high)})").plot_survival_function(ci_show=True)
KaplanMeierFitter().fit(low["OS_months"], low["Event"], label=f"Low (n={len(low)})").plot_survival_function(ci_show=True)
plt.xlabel("Overall Survival (Months)"); plt.ylabel("Survival Probability")
plt.title("GSE65858: Eustace 26-gene signature (full follow-up)", fontsize=11)
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
data["OS_months_5yr"] = np.minimum(data["OS_months"], 60.0)
data["Event_5yr"] = np.where(data["OS_months"] <= 60.0, data["Event"], 0)
d5 = data[["OS_months_5yr", "Event_5yr", "EustaceScore_c"]].dropna()
high5, low5 = d5[d5["EustaceScore_c"] >= med], d5[d5["EustaceScore_c"] < med]
lr_5yr = logrank_test(high5["OS_months_5yr"], low5["OS_months_5yr"], high5["Event_5yr"], low5["Event_5yr"])
print("Log-rank p (5-yr censored):", lr_5yr.p_value)

plt.figure(figsize=(7, 5.5))
KaplanMeierFitter().fit(high5["OS_months_5yr"], high5["Event_5yr"], label=f"High (n={len(high5)})").plot_survival_function(ci_show=True)
KaplanMeierFitter().fit(low5["OS_months_5yr"], low5["Event_5yr"], label=f"Low (n={len(low5)})").plot_survival_function(ci_show=True)
plt.xlabel("Overall Survival (Months)"); plt.ylabel("Survival Probability")
plt.title("GSE65858: Eustace 26-gene signature (5-yr censored)", fontsize=11)
plt.xlim(0, 60); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()